# 00 · Entorno, librerías y conexiones

Qué hace. Revisa que el computador tenga todo lo necesario para reproducir el proyecto: versión de Python y hardware,
librerías (con sus versiones, que quedan en `requirements.txt`), estructura de carpetas, datos de la competencia y, si se
pide, las conexiones con los servicios externos que se usaron para descargar datos.

Cómo. Importa cada librería y lee su versión; cuenta las filas de los archivos de Kaggle; revisa las carpetas. Las
conexiones externas (Earth Engine, OpenStreetMap, mapas base de Esri, descarga de NLTK) solo se prueban con
`PROBAR_CONEXIONES = True`: los cuadernos 01 a 04 no las necesitan, porque los datos descargados ya están en `stores/`.

Para qué. Es el primer cuaderno. Si todo sale CORRECTO, los demás deberían correr en orden sin errores.

| Cuaderno | Contenido |
|---|---|
| 00 | Entorno, librerías y conexiones |
| 01 | Análisis espacial descriptivo y texto como datos (limpieza, ortografía con BETO, DTM, TF-IDF, LSA, LDA, embeddings) |
| 02 | Modelo final: 162 variables, LightGBM, ×1,08, gemelos y envío |
| 03 | Validaciones (seis esquemas) y comparación de las seis familias |
| 04 | Errores del modelo y simulación de compras |

Conceptos de clase que usa: sistemas de referencia y EPSG:9377 para medir en metros (Lecture 14, diap. 10 y 11; lectura
Datos espaciales), extracción de variables de OpenStreetMap (lectura Datos espaciales, sección De las geometrías a
predictores), stopwords y stemming Snowball en español (Lecture 15, diap. 16 a 21) y BETO en GPU (Lecture 15, diap. 93 y 94).

| Conexión | Para qué se usó |
|---|---|
| Datos de Kaggle (local) | `train.csv`, `test.csv`, `submission_template.csv` de la competencia |
| API de Kaggle (opcional) | Enviar predicciones desde la terminal (no se usó: los envíos se hicieron desde la página) |
| Google Earth Engine | Capas satelitales: elevación, pendiente, vegetación, luces nocturnas, altura construida y población |
| OpenStreetMap: BBBike | Extracto `.pbf` de Bogotá con amenidades, transporte y vías |
| OpenStreetMap: Nominatim | Polígonos de localidades |
| OpenStreetMap: Overpass | Consultas puntuales a OSM (alternativa a BBBike) |
| Mapas base (Esri) | Fondo opcional de los mapas del cuaderno 01 |
| NLTK | Stopwords en español y stemmer para el texto de los avisos |
| GPU (CUDA) | Solo para recalcular la corrección ortográfica y los embeddings de BETO en el cuaderno 01 |

Tiempo: menos de un minuto. Al final hay una tabla con el estado de cada componente.

## 1. Sistema y hardware

In [1]:
# Banderas. PROBAR_CONEXIONES: True prueba los servicios externos (requiere internet y credenciales de Earth Engine).
# RECALCULAR: True instala con pip las librerías que falten. Con False solo se informa.
PROBAR_CONEXIONES = False
RECALCULAR = False

import sys
import os
import platform
import subprocess
import importlib
from pathlib import Path

ESTADO = {}   # resumen final: componente -> (estado, detalle)


def registrar(componente, ok, detalle="", si_falla="FALLA"):
    """Guarda el estado de un componente. `si_falla` permite marcar como opcional o con alternativa.
    ok = None: la prueba no se hizo en esta ejecución (PROBAR_CONEXIONES = False)."""
    estado = "NO PROBADA" if ok is None else ("CORRECTO" if ok else si_falla)
    ESTADO[componente] = (estado, detalle)
    print(f"[{estado}] {componente}: {detalle}")


print("Python      :", sys.version.split()[0], "|", sys.executable)
print("Sistema     :", platform.platform())
print("Procesador  :", platform.processor(), "| núcleos lógicos:", os.cpu_count())
try:
    import psutil
    mem = psutil.virtual_memory()
    print(f"RAM         : {mem.total / 1e9:.1f} GB total | {mem.available / 1e9:.1f} GB disponibles")
except ImportError:
    print("RAM         : (instalar psutil para verla)")
print("Directorio  :", Path.cwd())

Python      : 3.13.5 | <python de miniconda>
Sistema     : Windows-11-10.0.26300-SP0
Procesador  : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel | núcleos lógicos: 20
RAM         : 25.5 GB total | 10.8 GB disponibles
Directorio  : <carpeta del proyecto>\scripts


## 2. Librerías

Se listan todas las librerías que usa el proyecto. Si alguna falta y `RECALCULAR = True`, se instala con `pip`. PyTorch se
instala con soporte CUDA 12.8 (necesario para la GPU NVIDIA RTX 5060, arquitectura Blackwell); para los cuadernos 02 a 04
basta la versión para CPU.

In [2]:
# nombre para importar -> nombre en pip
LIBRERIAS = {
    # Datos y cálculo
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "pyarrow": "pyarrow", "joblib": "joblib",
    # Gráficos
    "matplotlib": "matplotlib", "seaborn": "seaborn",
    # Datos espaciales
    "geopandas": "geopandas", "shapely": "shapely", "pyproj": "pyproj", "osmnx": "osmnx", "osmium": "osmium",
    "contextily": "contextily", "folium": "folium", "mapclassify": "mapclassify", "libpysal": "libpysal", "esda": "esda",
    # Fuentes externas
    "ee": "earthengine-api", "requests": "requests",
    # Texto
    "nltk": "nltk",
    # Modelos (familias exigidas por el taller)
    "sklearn": "scikit-learn", "statsmodels": "statsmodels", "xgboost": "xgboost", "lightgbm": "lightgbm",
    "catboost": "catboost", "torch": "torch", "optuna": "optuna",
    # Texto con transformers (BETO, cuaderno 01)
    "transformers": "transformers",
    # Cuadernos y sistema
    "nbformat": "nbformat", "nbconvert": "nbconvert", "ipykernel": "ipykernel", "psutil": "psutil",
}


from importlib.metadata import version as version_pip, PackageNotFoundError


def version_de(modulo):
    """Importa el módulo (falla si no está) y devuelve la versión registrada por pip."""
    m = importlib.import_module(modulo)
    try:
        return version_pip(LIBRERIAS[modulo])
    except PackageNotFoundError:
        return getattr(m, "__version__", "instalada")


faltantes = []
for modulo, pip_nombre in LIBRERIAS.items():
    try:
        version_de(modulo)
    except ImportError:
        faltantes.append(pip_nombre)

for pip_nombre in (faltantes if RECALCULAR else []):
    print("Instalando", pip_nombre, "...")
    if pip_nombre == "torch":
        cmd = [sys.executable, "-m", "pip", "install", "torch", "--index-url", "https://download.pytorch.org/whl/cu128"]
    else:
        cmd = [sys.executable, "-m", "pip", "install", pip_nombre]
    subprocess.run(cmd, check=False)

import pandas as pd
versiones = []
for modulo, pip_nombre in LIBRERIAS.items():
    try:
        versiones.append({"librería (pip)": pip_nombre, "módulo": modulo, "versión": str(version_de(modulo))})
    except ImportError:
        versiones.append({"librería (pip)": pip_nombre, "módulo": modulo, "versión": "NO INSTALADA"})
versiones = pd.DataFrame(versiones)
registrar("Librerías", (versiones["versión"] != "NO INSTALADA").all(),
          f"{(versiones['versión'] != 'NO INSTALADA').sum()} de {len(versiones)} disponibles"
          + (f" | faltan: {faltantes}" if faltantes else ""))
versiones

[CORRECTO] Librerías: 32 de 32 disponibles


,librería (pip),módulo,versión
0,numpy,numpy,2.3.2
1,pandas,pandas,2.3.1
2,scipy,scipy,1.16.3
3,pyarrow,pyarrow,21.0.0
4,joblib,joblib,1.5.2
5,matplotlib,matplotlib,3.10.5
6,seaborn,seaborn,0.13.2
7,geopandas,geopandas,1.1.4
8,shapely,shapely,2.1.2
9,pyproj,pyproj,3.8.0


In [3]:
# requirements.txt en la raíz del proyecto (para reproducir el entorno desde el repositorio)
RAIZ = Path("..").resolve()
lineas = [f"{r['librería (pip)']}=={r['versión'].split('+')[0]}" for _, r in versiones.iterrows()
          if r["versión"] not in ("NO INSTALADA", "instalada")]
(RAIZ / "requirements.txt").write_text(
    "# Generado por scripts/00_entorno_y_conexiones.ipynb\n"
    "# PyTorch con CUDA 12.8: pip install torch --index-url https://download.pytorch.org/whl/cu128\n"
    + "\n".join(lineas) + "\n", encoding="utf-8")
print(f"requirements.txt con {len(lineas)} librerías en {RAIZ}")

requirements.txt con 32 librerías en <carpeta del proyecto>


## 3. Estructura de carpetas

| Carpeta | Contenido |
|---|---|
| `Kaggle/` | Datos de la competencia (no se modifican) |
| `scripts/` | Cuadernos numerados en orden de ejecución, scripts del flujo final y `LEEME.md` |
| `stores/` | Datos descargados de fuentes externas, bases intermedias y resultados de validación |
| `stores/recalculo/` | Solo se crea si un cuaderno corre con `RECALCULAR = True` |
| `views/` | Figuras de los cuadernos, con el prefijo del cuaderno |
| `submissions/kaggle_subidos/` | Envíos a Kaggle archivados en solo lectura, con su registro |
| `document/` | Presentaciones (`best_equipo_02`, `compare_equipo_02`) |

La celda solo revisa que existan; no crea ni modifica nada.

In [4]:
CARPETAS = ["Kaggle", "scripts", "stores", "stores/osm", "stores/texto", "stores/catastro", "views",
            "submissions/kaggle_subidos", "document/presentaciones"]
pd.DataFrame({"carpeta": CARPETAS, "existe": [(RAIZ / c).is_dir() for c in CARPETAS],
              "archivos": [len([f for f in (RAIZ / c).iterdir() if f.is_file()]) if (RAIZ / c).is_dir() else 0 for c in CARPETAS]})

,carpeta,existe,archivos
0,Kaggle,True,3
1,scripts,True,26
2,stores,True,45
3,stores/osm,True,3
4,stores/texto,True,12
5,stores/catastro,True,5
6,views,True,44
7,submissions/kaggle_subidos,True,1
8,document/presentaciones,True,36


## 4. Conexiones

### 4.1 Datos de la competencia (Kaggle)

In [5]:
ARCHIVOS_KAGGLE = {"train.csv": 38644, "test.csv": 10286, "submission_template.csv": 10286}
ok_kaggle = True
for nombre, filas_esperadas in ARCHIVOS_KAGGLE.items():
    ruta = RAIZ / "Kaggle" / nombre
    if ruta.exists():
        filas = sum(1 for _ in open(ruta, encoding="utf-8")) - 1
        print(f"  {nombre}: {filas:,} filas (esperadas {filas_esperadas:,})")
        ok_kaggle &= filas == filas_esperadas
    else:
        print(f"  {nombre}: NO ENCONTRADO")
        ok_kaggle = False
registrar("Datos de Kaggle (local)", ok_kaggle, "competencia uniandes-bdml-202620-ps-3")

# API de Kaggle (opcional): requiere ~/.kaggle/kaggle.json o las variables KAGGLE_USERNAME / KAGGLE_KEY
token = Path.home() / ".kaggle" / "kaggle.json"
api_ok = token.exists() or ("KAGGLE_USERNAME" in os.environ and "KAGGLE_KEY" in os.environ)
registrar("API de Kaggle (opcional)", api_ok,
          "token encontrado" if api_ok else "sin token: los envíos se hacen desde la página web de la competencia",
          si_falla="NO CONFIGURADA (opcional)")

  train.csv: 38,644 filas (esperadas 38,644)
  test.csv: 10,286 filas (esperadas 10,286)
  submission_template.csv: 10,286 filas (esperadas 10,286)
[CORRECTO] Datos de Kaggle (local): competencia uniandes-bdml-202620-ps-3
[NO CONFIGURADA (opcional)] API de Kaggle (opcional): sin token: los envíos se hacen desde la página web de la competencia


### 4.2 Google Earth Engine

Se reutiliza la conexión de su proyecto de Google Cloud registrado en Earth Engine, que se escribe en `EE_PROYECTO` (credenciales en `~/.config/earthengine/credentials`).
Si la inicialización falla, ejecutar una vez `ee.Authenticate()`. Las capas descargadas están en `stores/gee_features.parquet`.
Última prueba correcta: 23 de noviembre de 2026 (elevación de prueba en Chapinero: 2.585 m).

In [6]:
import ee
EE_PROYECTO = ""  # escriba aquí el ID de su proyecto de Google Cloud registrado en Earth Engine
if PROBAR_CONEXIONES:
    try:
        ee.Initialize(project=EE_PROYECTO)
        punto = ee.Geometry.Point([-74.06, 4.65])                     # Chapinero
        elev = ee.Image("USGS/SRTMGL1_003").reduceRegion(ee.Reducer.first(), punto, 30).getInfo()["elevation"]
        registrar("Google Earth Engine", True, f"proyecto {EE_PROYECTO} | elevación de prueba en Chapinero: {elev} m")
    except Exception as e:
        registrar("Google Earth Engine", False, f"{type(e).__name__}: {str(e)[:150]}; correr ee.Authenticate()")
else:
    registrar("Google Earth Engine", None, "datos ya descargados en stores/gee_features.parquet")

[NO PROBADA] Google Earth Engine: datos ya descargados en stores/gee_features.parquet


### 4.3 OpenStreetMap

1. BBBike: extracto `.pbf` de Bogotá (fuente principal; una sola descarga reproducible, en `stores/osm/`).
2. Nominatim: geocodificación de polígonos (localidades) con `osmnx`; resultado en `stores/localidades_bogota.geojson`.
3. Overpass: consultas puntuales. Los servidores públicos se saturan con frecuencia, así que se prueban varios.

In [7]:
if PROBAR_CONEXIONES:
    import requests
    CABECERA = {"User-Agent": "bdml-ps3-academico"}

    try:
        r = requests.head("https://download.bbbike.org/osm/bbbike/Bogota/Bogota.osm.pbf", timeout=30,
                          headers=CABECERA, allow_redirects=True)
        registrar("OSM · BBBike", r.status_code == 200,
                  f"HTTP {r.status_code} | {int(r.headers.get('Content-Length', 0)) / 1e6:.1f} MB | "
                  f"actualizado {r.headers.get('Last-Modified', '?')}")
    except Exception as e:
        registrar("OSM · BBBike", False, f"{type(e).__name__}: {str(e)[:120]}")

    try:
        import osmnx as ox
        ox.settings.use_cache = True
        ox.settings.cache_folder = str(RAIZ / "scripts" / "cache")
        g = ox.geocode_to_gdf("Localidad Chapinero, Bogotá, Colombia")
        registrar("OSM · Nominatim", len(g) == 1, f"polígono de Chapinero recibido (osm_id {int(g['osm_id'].iloc[0])})")
    except Exception as e:
        registrar("OSM · Nominatim", False, f"{type(e).__name__}: {str(e)[:120]}")

    SERVIDORES_OVERPASS = ["https://overpass.private.coffee/api/interpreter", "https://overpass-api.de/api/interpreter",
                           "https://overpass.kumi.systems/api/interpreter"]
    consulta = '[out:json][timeout:20];node["public_transport"="station"](4.64,-74.07,4.68,-74.04);out tags 3;'
    disponibles = []
    for url in SERVIDORES_OVERPASS:
        try:
            r = requests.post(url, data={"data": consulta}, timeout=30, headers=CABECERA)
            if r.status_code == 200 and "elements" in r.text:
                disponibles.append(url.split("/")[2])
        except Exception:
            pass
    registrar("OSM · Overpass", len(disponibles) > 0,
              f"servidores disponibles: {disponibles}" if disponibles else "todos saturados en este momento",
              si_falla="NO DISPONIBLE (alternativa: BBBike)")
else:
    for comp, det in [("OSM · BBBike", "extracto en stores/osm/Bogota.osm.pbf"),
                      ("OSM · Nominatim", "localidades en stores/localidades_bogota.geojson"),
                      ("OSM · Overpass", "no se usa para los datos finales")]:
        registrar(comp, None, det)

[NO PROBADA] OSM · BBBike: extracto en stores/osm/Bogota.osm.pbf
[NO PROBADA] OSM · Nominatim: localidades en stores/localidades_bogota.geojson
[NO PROBADA] OSM · Overpass: no se usa para los datos finales


### 4.4 Mapas base (Esri World Gray Canvas)

Se usa Esri porque CARTO ahora exige clave de API. Es opcional: el cuaderno 01 solo lo agrega con `USAR_MAPA_BASE = True`.

In [8]:
import contextily as cx
if PROBAR_CONEXIONES:
    try:
        url = cx.providers.Esri.WorldGrayCanvas.build_url(x=1204, y=1993, z=12)
        r = requests.get(url, timeout=30, headers=CABECERA)
        registrar("Mapas base (Esri)", r.status_code == 200, f"HTTP {r.status_code} | tesela de {len(r.content) / 1e3:.0f} KB")
    except Exception as e:
        registrar("Mapas base (Esri)", False, f"{type(e).__name__}: {str(e)[:120]}")
else:
    registrar("Mapas base (Esri)", None, "opcional; los mapas se dibujan sin fondo")

[NO PROBADA] Mapas base (Esri): opcional; los mapas se dibujan sin fondo


### 4.5 NLTK: stopwords en español y stemmer Snowball

In [9]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
try:
    try:
        sw = stopwords.words("spanish")                    # copia local
    except LookupError:
        if not PROBAR_CONEXIONES:
            raise
        nltk.download("stopwords", quiet=True)             # descarga solo si falta y se permite internet
        sw = stopwords.words("spanish")
    registrar("NLTK (español)", len(sw) > 0,
              f"{len(sw)} stopwords | stem('remodelado') = '{SnowballStemmer('spanish').stem('remodelado')}'")
except Exception as e:
    registrar("NLTK (español)", False, f"{type(e).__name__}: falta el corpus stopwords; correr nltk.download('stopwords')")

[CORRECTO] NLTK (español): 313 stopwords | stem('remodelado') = 'remodel'


### 4.6 GPU (CUDA) para PyTorch

Solo hace falta para recalcular la corrección ortográfica y los embeddings de BETO en el cuaderno 01
(`RECALCULAR = True`). Los cuadernos 02 a 04 corren en CPU. El portátil del equipo (Lenovo, modo Híbrido) apaga la GPU
dedicada cuando funciona con batería, y la GPU no es visible desde terminales restringidas. Aquí solo se consulta si está
disponible; no se hace ningún cálculo en ella.

In [10]:
import torch
if torch.cuda.is_available():
    registrar("GPU (CUDA)", True, f"{torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda} | "
              f"capacidad {torch.cuda.get_device_capability(0)}")
else:
    registrar("GPU (CUDA)", False, f"PyTorch {torch.__version__} no ve una GPU; solo se necesita para recalcular BETO",
              si_falla="NO DISPONIBLE (opcional)")

[CORRECTO] GPU (CUDA): NVIDIA GeForce RTX 5060 Laptop GPU | CUDA 12.8 | capacidad (12, 0)


## 5. Resumen del entorno

In [11]:
resumen = pd.DataFrame([{"componente": k, "estado": v[0], "detalle": v[1]} for k, v in ESTADO.items()])
pd.set_option("display.max_colwidth", 140)
resumen

,componente,estado,detalle
0,Librerías,CORRECTO,32 de 32 disponibles
1,Datos de Kaggle (local),CORRECTO,competencia uniandes-bdml-202620-ps-3
2,API de Kaggle (opcional),NO CONFIGURADA (opcional),sin token: los envíos se hacen desde la página web de la competencia
3,Google Earth Engine,NO PROBADA,datos ya descargados en stores/gee_features.parquet
4,OSM · BBBike,NO PROBADA,extracto en stores/osm/Bogota.osm.pbf
5,OSM · Nominatim,NO PROBADA,localidades en stores/localidades_bogota.geojson
6,OSM · Overpass,NO PROBADA,no se usa para los datos finales
7,Mapas base (Esri),NO PROBADA,opcional; los mapas se dibujan sin fondo
8,NLTK (español),CORRECTO,313 stopwords | stem('remodelado') = 'remodel'
9,GPU (CUDA),CORRECTO,"NVIDIA GeForce RTX 5060 Laptop GPU | CUDA 12.8 | capacidad (12, 0)"
